In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from google.colab import userdata
import hashlib, json, os, pathlib, subprocess, sys, tempfile, uuid, zipfile
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['CEG_WM_ROOT_KEY'] = userdata.get('CEG_WM_ROOT_KEY')
if not os.environ['HF_TOKEN'] or not os.environ['CEG_WM_ROOT_KEY']: raise RuntimeError('required Colab userdata secrets are unavailable')
os.environ['CEG_WM_CACHE_ROOT'] = '/content/drive/MyDrive/CEG-WM/cache'
os.environ['CEG_WM_PERSISTENT_ROOT'] = '/content/drive/MyDrive/CEG-WM/models'
workspace = pathlib.Path(tempfile.mkdtemp(prefix='ceg-wm-stage-a-', dir='/content'))
repository = workspace / 'repository'
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'main', '--single-branch', 'https://github.com/RICHAAARC/CEG-WM.git', str(repository)], check=True)
revision = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=repository, check=True, text=True, capture_output=True).stdout.strip()
archive = workspace / 'candidate.zip'
subprocess.run([sys.executable, str(repository / 'scripts/experiment_execution/build_contrastive_lf_branch_attribution_package.py'), '--repository-root', str(repository), '--source-revision', revision, '--output', str(archive)], check=True)
handoff = json.loads(archive.with_suffix('.zip.manifest.json').read_text())
package_sha256 = hashlib.sha256(archive.read_bytes()).hexdigest()
if package_sha256 != handoff['archive_sha256']: raise RuntimeError('locally built package digest mismatch')
extract_root = workspace / 'package'
extract_root.mkdir(mode=0o700)
with zipfile.ZipFile(archive) as source: source.extractall(extract_root)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(extract_root / 'requirements_semantic_texture_operational_preflight.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'Pillow==12.3.0'], check=True)
new_run_id = 'contrastive-lf-branch-attribution-' + uuid.uuid4().hex
session_id = 'stage-a-session-' + uuid.uuid4().hex
runs_root = pathlib.Path('/content/drive/MyDrive/CEG-WM/contrastive_lf_branch_attribution_validation/candidate_selection')
runs_root.mkdir(parents=True, exist_ok=True)
print(json.dumps({'resolved_repository_revision': revision, 'new_run_id_if_needed': new_run_id, 'session_id': session_id}, sort_keys=True))


In [ ]:
command = [sys.executable, str(extract_root / 'scripts/experiment_execution/contrastive_lf_branch_attribution_bootstrap.py'), '--expected-revision', revision, '--expected-package-identity', handoff['package_identity'], '--expected-embedded-manifest-sha256', handoff['embedded_manifest_sha256'], '--new-run-id', new_run_id, '--session-id', session_id, '--runs-root', str(runs_root), '--package-sha256', package_sha256]
completed = subprocess.run(command, check=False, text=True)
if completed.returncode not in (0, 2, 3): raise RuntimeError('Stage-A launcher returned an unsupported code')
session_receipts = list(runs_root.glob('*/sessions/' + session_id + '.json'))
if len(session_receipts) != 1: raise RuntimeError('authenticated session receipt is missing or ambiguous')
session = json.loads(session_receipts[0].read_text())
run_root = session_receipts[0].parents[1]
final_root = run_root / 'final'
if completed.returncode in (0, 2):
    receipt = json.loads((final_root / 'contrastive_lf_execution_receipt.json').read_text())
    result = json.loads((final_root / receipt['result_filename']).read_text())
    for row in (final_root / 'SHA256SUMS').read_text().splitlines():
        digest, filename = row.split('  ', 1)
        if hashlib.sha256((final_root / filename).read_bytes()).hexdigest() != digest: raise RuntimeError('delivery checksum mismatch')
    print(json.dumps({'returncode': completed.returncode, 'result_classification': result['result_classification'], 'run_id': run_root.name, 'candidate_selection_passed': result['candidate_selection_passed'], 'producer_revisions': result['producer_revisions']}, sort_keys=True))
else:
    snapshot_path = pathlib.Path(session['most_recent_snapshot_path'])
    snapshot_sums = snapshot_path.with_name(snapshot_path.stem + '.SHA256SUMS')
    for row in snapshot_sums.read_text().splitlines():
        digest, filename = row.split('  ', 1)
        if hashlib.sha256((snapshot_path.parent / filename).read_bytes()).hexdigest() != digest: raise RuntimeError('progress snapshot checksum mismatch')
    print(json.dumps({'returncode': 3, 'session_status': session['session_status'], 'run_id': run_root.name, 'most_recent_snapshot_path': str(snapshot_path), 'committed_unit_count': session['committed_unit_count']}, sort_keys=True))
